In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os

ANNOT_FILE = "data/dataset_092624.xlsx"

In [3]:
# Load the xlsx file into a dataframe
annotations = pd.read_excel(ANNOT_FILE)

# Display the head of the dataframe
print(annotations.head())

# Filter only relevant rows : valid_yn = 'yes' and source = 'dryad'
annotations = annotations[(annotations['valid_yn'] == 'yes') & (annotations['source'] == 'dryad')]
annotations.head()

   id                                      url  \
0   3  https://doi.org/10.5061/dryad.05qfttf0n   
1   4  https://doi.org/10.5061/dryad.0rxwdbs2c   
2   5      https://doi.org/10.5061/dryad.121sk   
3   7  https://doi.org/10.5061/dryad.12jm63xtw   
4   9      https://doi.org/10.5061/dryad.1771t   

                                               title  \
0  Fish mock community with 41 species from 13 or...   
1  Exploration and diet specialization in eastern...   
2  Data from: Paternity analysis of wood turtles ...   
3  Data from: 60 specific eDNA qPCR assays to det...   
4  Data from: Resampling method for applying dens...   

                                           full_text publication_year  source  \
0  Fish mock community with 41 species from 13 or...             2020  zenodo   
1  Exploration and diet specialization in eastern...             2022  zenodo   
2  Data from: Paternity analysis of wood turtles ...             2017  zenodo   
3  Data from: 60 specific eDNA qPCR as

,id,url,title,full_text,publication_year,source,id_query,reason_non_valid,valid_yn,dataset_relevance,...,threatened_species,new_species_science,new_species_region,bias_north_south,url.1,MC_dataset_type,MC_spatial_range,MC_temporal_range,MC_relevance,MC_relevance_modifiers
4,9,https://doi.org/10.5061/dryad.1771t,Data from: Resampling method for applying dens...,Data from: Resampling method for applying dens...,2016,dryad,"0,3,4,8,10",NaN,yes,L,...,False,False,False,False,https://doi.org/10.5061/dryad.1771t,H,X,L,L,L
8,13,https://doi.org/10.5061/dryad.24rj8,"Data from: Aspicilia bicensis (Megasporaceae),...","Data from: Aspicilia bicensis (Megasporaceae),...",2016,dryad,"3,5,8",NaN,yes,M,...,False,True,False,False,https://doi.org/10.5061/dryad.24rj8,L,X,X,X,L
9,16,https://doi.org/10.5061/dryad.2b8f1,Data from: Do genetic drift and accumulation o...,Data from: Do genetic drift and accumulation o...,2017,dryad,"3,4,6",NaN,yes,H,...,False,False,False,True,https://doi.org/10.5061/dryad.2b8f1,H,H,M,H,H
11,19,https://doi.org/10.5061/dryad.2n5h6,Data from: The genetic signature of range expa...,Data from: The genetic signature of range expa...,2016,dryad,"9,7,8,3,6",NaN,yes,M,...,False,False,False,False,https://doi.org/10.5061/dryad.2n5h6,H,L,M,M,M
15,27,https://doi.org/10.5061/dryad.3nh72,Data from: Patchy distribution and low effecti...,Data from: Patchy distribution and low effecti...,2017,dryad,"3,6",NaN,yes,M,...,True,False,False,False,https://doi.org/10.5061/dryad.3nh72,H,X,M,L,M


In [12]:
from llm_metadata.schemas.validation import DataFrameValidator
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction

# Validate annotations dataframe using the reusable DataFrameValidator
validator = DataFrameValidator(DatasetFeatureExtraction)
report = validator.validate(annotations)

print(report.summary())

if report.errors:
    print("\nSample Errors:")
    display(report.errors_to_dataframe().head())

# Replace annotations for downstream steps if validation succeeded
if report.valid_rows:
    # Get the normalized/validated data
    clean_df = report.valid_rows_to_dataframe()
    
    # Merge: Keep original columns but overwrite with validated/normalized values
    # Filter original df to valid rows first
    annotations_valid = annotations.loc[report.valid_indices].copy()
    
    # Update columns from the validated model (handles normalization of lists, enums etc)
    for col in clean_df.columns:
        annotations_valid[col] = clean_df[col]
        
    annotations = annotations_valid
    
print(f"\nProceeding with {len(annotations)} valid rows.")
annotations.head()

2026-01-07 08:52:46.750 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 37 valid, 0 invalid, 0 errors


VALIDATION REPORT
Total rows:     37
Valid rows:     37 (100.0%)
Invalid rows:   0

Error Breakdown:
  Type/Schema errors: 0
  Data/Value errors:  0

Errors by Field:

Proceeding with 37 valid rows.


,id,url,title,full_text,publication_year,source,id_query,reason_non_valid,valid_yn,dataset_relevance,...,new_species_science,new_species_region,bias_north_south,url.1,MC_dataset_type,MC_spatial_range,MC_temporal_range,MC_relevance,MC_relevance_modifiers,reason_not_valid
__orig_index__,,,,,,,,,,,,,,,,,,,,,
4,9,https://doi.org/10.5061/dryad.1771t,Data from: Resampling method for applying dens...,Data from: Resampling method for applying dens...,2016,dryad,"0,3,4,8,10",NaN,yes,L,...,False,False,False,https://doi.org/10.5061/dryad.1771t,H,X,L,L,L,None
8,13,https://doi.org/10.5061/dryad.24rj8,"Data from: Aspicilia bicensis (Megasporaceae),...","Data from: Aspicilia bicensis (Megasporaceae),...",2016,dryad,"3,5,8",NaN,yes,M,...,True,False,False,https://doi.org/10.5061/dryad.24rj8,L,X,X,X,L,None
9,16,https://doi.org/10.5061/dryad.2b8f1,Data from: Do genetic drift and accumulation o...,Data from: Do genetic drift and accumulation o...,2017,dryad,"3,4,6",NaN,yes,H,...,False,False,True,https://doi.org/10.5061/dryad.2b8f1,H,H,M,H,H,None
11,19,https://doi.org/10.5061/dryad.2n5h6,Data from: The genetic signature of range expa...,Data from: The genetic signature of range expa...,2016,dryad,"9,7,8,3,6",NaN,yes,M,...,False,False,False,https://doi.org/10.5061/dryad.2n5h6,H,L,M,M,M,None
15,27,https://doi.org/10.5061/dryad.3nh72,Data from: Patchy distribution and low effecti...,Data from: Patchy distribution and low effecti...,2017,dryad,"3,6",NaN,yes,M,...,False,False,False,https://doi.org/10.5061/dryad.3nh72,H,X,M,L,M,None


In [5]:
from llm_metadata.prefect_pipeline import doi_classification_pipeline

# Run classification on the first 3 validated rows
# Use `annotations` which will be replaced by `annotations_valid` if validation succeeded

dois = annotations['url'].head(3)
dois = [doi.replace('https://doi.org/', '') for doi in dois]

classification_results = doi_classification_pipeline(dois=dois)

08:37:47.675 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8367
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

08:37:54.239 | INFO    | Flow run 'expert-mamba' - Beginning flow run 'expert-mamba' for flow 'doi-classification-pipeline'

08:37:56.335 | INFO    | Task run 'fetch_abstracts-d38' - Finished in state Completed()

08:38:00.026 | INFO    | Task run 'classify_abstract_task-263' - Finished in state Completed()

08:38:00.058 | INFO    | Task run 'classify_abstract_task-e48' - Finished in state Completed()

08:38:00.263 | INFO    | Task run 'classify_abstract_task-485' - Finished in state Completed()

08:38:00.285 | INFO    | Flow run 'expert-mamba' - Finished in state Completed()

In [6]:
classification_results[0]['output'].model_dump()

{'data_type': ['genetic_analysis', 'traits'],
 'geospatial_info_dataset': ['site', 'sample'],
 'spatial_range_km2': None,
 'temporal_range': None,
 'temp_range_i': None,
 'temp_range_f': None,
 'species': ['Salvelinus namaycush'],
 'referred_dataset': 'Québec, Canada',
 'valid_yn': 'yes',
 'reason_not_valid': None}

In [ ]:
outputs = [r['output'].model_dump() for r in classification_results]
for d, doi in zip(outputs, dois):
    d.update({'doi': doi})

# Parse outputs to pandas
output_df = pd.DataFrame(outputs)

# Input df with only attributes that are present in both the annotations and the output
common_cols = annotations.columns.intersection(output_df.columns)
compared_df = annotations[common_cols].head()

In [ ]:
compared_df = annotations[['title', 'url', 'data_type', 'geospatial_info_dataset', 'spatial_range_km2', 'temporal_range', 'temp_range_i', 'temp_range_f', 'species', 'referred_dataset']].head(3)
compared_df['doi'] = compared_df['url'].apply(lambda x: x.replace('https://doi.org/', ''))

# Ensure data_type is a list (it might already be a list after validation)

compared_df['classification'] = 'manual'
output_df['classification'] = 'automated'

# Append the output_df to the compared_df
compared_df = pd.concat([compared_df, output_df], ignore_index=True, sort=False)

# Reorder the columns to put doi first
reorder_columns = [c for c in compared_df.columns.tolist() if c not in ('title', 'url', 'doi', 'classification')]
compared_df = compared_df[['title', 'url', 'doi', 'classification'] + reorder_columns]

# Order by doi, classification
compared_df = compared_df.sort_values(by=['doi', 'classification'], ascending=[True, False])
compared_df


C:\Users\beav3503\AppData\Local\Temp\ipykernel_25604\1288748570.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  compared_df = pd.concat([compared_df, output_df], ignore_index=True, sort=False)


,title,url,doi,classification,data_type,geospatial_info_dataset,spatial_range_km2,temporal_range,temp_range_i,temp_range_f,taxons,referred_dataset,species,valid_yn,reason_not_valid
0,Data from: Resampling method for applying dens...,https://doi.org/10.5061/dryad.1771t,10.5061/dryad.1771t,manual,[density],[sample],NaN,2008,2008.0,2008.0,"[raccoons, striped skunks]",None,NaN,NaN,NaN
3,NaN,NaN,10.5061/dryad.1771t,automated,"[genetic_analysis, traits]","[site, sample]",NaN,None,NaN,NaN,NaN,"Québec, Canada",[Salvelinus namaycush],yes,None
1,"Data from: Aspicilia bicensis (Megasporaceae),...",https://doi.org/10.5061/dryad.24rj8,10.5061/dryad.24rj8,manual,[genetic_analysis],None,NaN,None,NaN,NaN,[Aspicilia bicensis],None,NaN,NaN,NaN
4,NaN,NaN,10.5061/dryad.24rj8,automated,"[abundance, density, distribution]","[sample, site, distribution]",NaN,None,NaN,NaN,NaN,None,"[raccoons, striped skunks]",yes,None
2,Data from: Do genetic drift and accumulation o...,https://doi.org/10.5061/dryad.2b8f1,10.5061/dryad.2b8f1,manual,[genetic_analysis],None,500000.0,2003-2013,2003.0,2013.0,[Salvelinus namaycush],None,NaN,NaN,NaN
5,NaN,NaN,10.5061/dryad.2b8f1,automated,"[traits, distribution]","[site, distribution]",NaN,None,NaN,NaN,NaN,"Parc national du Bic, Quebec, Canada",[Aspicilia bicensis],yes,None


In [11]:
def compare_label(truth, pred):
    if truth and truth == pred:
        return 'TP'
    elif truth and truth != pred:
        return 'FN'
    elif not truth and pred:
        return 'FP'
    else:
        return 'TN'

# Fuzzy matching
from fuzzywuzzy import fuzz
def fuzzy_compare_label(truth, pred):
    # Normalize missing values (pandas uses nan floats)
    if isinstance(truth, float) and pd.isna(truth):
        truth = None
    if isinstance(pred, float) and pd.isna(pred):
        pred = None

    if truth and pred:
        fuzz_ratio = fuzz.ratio(str(truth), str(pred))
        if fuzz_ratio > 80:
            return 'TP'
        else:
            return 'FN'
    elif truth and not pred:
        return 'FN'
    elif not truth and pred:
        return 'FP'
    else:
        return 'TN'
    
def compare_multilabel(truth: list, pred: list):
    out = []
    for p in pred:
        if truth and p in truth:
            out.append('TP')
        elif truth and p not in truth:
            out.append('FN')
        elif not truth and p:
            out.append('FP')
        else:
            out.append('TN')
    return out

FEATURE_COMPARE_FN = {
    'data_type': compare_multilabel,
    'geospatial_info_dataset': compare_label,
    'spatial_range_km2': compare_label,
    'temporal_range': compare_label,
    'temp_range_i': compare_label,
    'temp_range_f': compare_label,
    'taxons': fuzzy_compare_label,
    'referred_dataset': compare_label
}

def compare_doi_rows(df, doi) -> pd.Series:
    pred_df = df[(df['doi'] == doi) & (df['classification'] == 'automated')]
    truth_df = df[(df['doi'] == doi) & (df['classification'] == 'manual')]
    if pred_df.empty or truth_df.empty:
        return None

    pred_series = pred_df.iloc[0]
    truth_series = truth_df.iloc[0]

    out = {}
    for feature, compare_fn in FEATURE_COMPARE_FN.items():
        out[feature] = compare_fn(truth_series.get(feature), pred_series.get(feature))
    return pd.Series(out)

# Run a sample comparison
dois = compared_df['doi'].unique().tolist()

compare_doi_rows(compared_df, '10.5061/dryad.1771t')

data_type                  [FN, FN]
geospatial_info_dataset          FN
spatial_range_km2                FN
temporal_range                   FN
temp_range_i                     FN
temp_range_f                     FN
taxons                           FN
referred_dataset                 FP
dtype: object

In [69]:
compared_df['doi'].unique().tolist()

['10.5061/dryad.1771t', '10.5061/dryad.24rj8', '10.5061/dryad.2b8f1']